In [1]:
import pandas as pd
import sys

sys.path.insert(0, '..')
import functions.myfunctions as mf
import tidytcells as tt

In [2]:
vdj = pd.read_csv('../data/vdj_cleaned_subset_for_MI.csv', index_col = 0)
vdj = vdj.loc[vdj['Epitope'] != 'KLGGALQAK'] # because too big - takes forever
vdj = mf.prepare_data(vdj, col1 = 'cdr3a_IMGTgaps', col2='cdr3b_IMGTgaps', type = 'cdr3').drop_duplicates(subset = ['V-a','J-a','cdr3a_IMGTgaps_padded','V-b','J-b','cdr3b_IMGTgaps_padded', 'Epitope'])
print(vdj.shape)

(9785, 37)


In [3]:
for c in ['V-a', 'V-b', 'J-a', 'J-b']:
    print(c)
    vdj[c] = vdj.apply(
        lambda row: pd.NA if type(row[c]) != str else tt.tcr.standardise(
            gene_name=row[c],
            species=row['Species'],
            precision='gene'
        ),
        axis=1
    )
vdj = vdj.dropna(subset=['V-a', 'V-b', 'J-a', 'J-b'])
print('shape before tidytcells: ', vdj.shape, '; shape after tidytcells: ', vdj.shape)

V-a
V-b
J-a
J-b
shape before tidytcells:  (9785, 37) ; shape after tidytcells:  (9785, 37)


In [4]:
vdj = vdj.drop_duplicates(subset = ['V-a','J-a','cdr3a_IMGTgaps_padded','V-b','J-b','cdr3b_IMGTgaps_padded', 'Epitope'])
print(vdj.shape)

(9744, 37)


In [5]:
_10x_ref = 'https://www.10xgenomics.com/resources/application-notes/a-new-way-of-exploring-immunity-linking-highly-multiplexed-antigen-recognition-to-immune-repertoire-and-phenotype/#'
vdj['Reference'] = vdj['Reference'].replace(_10x_ref, '10xGenomics').fillna('unknown')

In [6]:
X = vdj[['Epitope','Reference']].value_counts()
print(X.shape)
X = X.loc[X > 2]
X = X.reset_index()

(78,)


In [7]:
X

,Epitope,Reference,0
0,AVFDRKSDAK,10xGenomics,1692
1,GILGFVFTL,10xGenomics,1222
2,RAKFKQLL,10xGenomics,1200
3,IVTDFSVIK,10xGenomics,697
4,RLRAEAQVK,10xGenomics,412
5,SSLENFRAYV,PMID:28636592,349
6,ELAGIGILTV,10xGenomics,325
7,HGIRNASFI,PMID:28636592,243
8,TTDPSFLGRY,https://github.com/antigenomics/vdjdb-db/issue...,242
9,ASNENMETM,PMID:28636592,196


In [8]:
vdjdb_no_small_study = []

for ep in vdj['Epitope'].unique():
    vdj_ep = vdj.loc[vdj['Epitope'] == ep]
    allowed_refs = X.loc[X['Epitope']== ep]['Reference'].tolist()
    print(allowed_refs)
    vdj_ep_large = vdj_ep.loc[vdj_ep['Reference'].isin(allowed_refs)]
    vdjdb_no_small_study.append(vdj_ep_large)

['10xGenomics', 'PMID:12555663', 'PMID:19786555']
['10xGenomics', 'PMID:28636592', 'PMID:28636589', 'PMID:29997621', 'PMID:34793243', 'PMID:29483513', 'PMID:27645996', 'PMID:28423320', 'PMID:7807026']
['https://doi.org/10.1101/2020.05.04.20085779', 'PMID:28636592', 'PMID:28636589', 'PMID:16237109', 'https://github.com/antigenomics/vdjdb-db/issues/252', 'PMID:34793243', 'PMID:28623251', 'PMID:28423320', 'PMID:28934479']
['10xGenomics', 'PMID:28636592', 'PMID:34793243', 'PMID:11046006', 'PMID:27645996', 'PMID:28636589']
['PMID:28636592']
['PMID:28636592']
['https://github.com/antigenomics/vdjdb-db/issues/326', 'PMID:34793243', 'PMID:33951417', 'PMID:33664060']
['PMID:28103239', 'PMID:28934479']
['unknown', 'PMID:26860370', 'PMID:27645996']
['unknown']
['unknown']
['10xGenomics', 'PMID:9207000']
['10xGenomics', 'PMID:9207000']
['10xGenomics']
['10xGenomics']
['PMID:28636592']
['PMID:28636592']
['PMID:28636592']
['PMID:28636592']
['https://github.com/antigenomics/vdjdb-db/issues/326', 'PMI

In [9]:
vdjdb_no_small_study = pd.concat(vdjdb_no_small_study)
vdjdb_no_small_study

,Unnamed: 0,complex.id,Gene-a,CDR3-a,V-a,J-a,Species,MHC A,MHC B,MHC class,...,cdr2a_IMGTgaps,cdr2b_IMGTgaps,cdr3a_IMGTgaps,cdr3b_IMGTgaps,len_cdr3a,len_cdr3b,len_cdr3a_IMGTgaps,cdr3a_IMGTgaps_padded,len_cdr3b_IMGTgaps,cdr3b_IMGTgaps_padded
0,13,14,TRA,CAYTVLGNEKLTF,TRAV38-1,TRAJ48,HomoSapiens,HLA-A*02,B2M,MHCI,...,QEAY--KQQN,SYD----VKM,CAYTVLG--NEKLTF,CASSFTP--YNEQFF,15,15,15,"C, A, Y, T, V, L, G, -, -, -, -, -, -, N, E, K...",15,"C, A, S, S, F, T, P, -, -, -, -, -, -, Y, N, E..."
1,14,15,TRA,CAVAGYGGSQGNLIF,TRAV12-2,TRAJ42,HomoSapiens,HLA-A*02,B2M,MHCI,...,IYS----NGD,SYD----VKM,CAVAGYGGSQGNLIF,CASSPQG-LGTEAFF,15,15,15,"C, A, V, A, G, Y, G, G, -, -, -, -, S, Q, G, N...",15,"C, A, S, S, P, Q, G, -, -, -, -, -, L, G, T, E..."
2,15,16,TRA,CAVSFGNEKLTF,TRAV12-2,TRAJ48,HomoSapiens,HLA-A*02,B2M,MHCI,...,IYS----NGD,SYD----VKM,CAVSFG---NEKLTF,CAEGQGF-VGQPQHF,15,15,15,"C, A, V, S, F, G, -, -, -, -, -, -, -, N, E, K...",15,"C, A, E, G, Q, G, F, -, -, -, -, -, V, G, Q, P..."
3,16,17,TRA,CAVTHYGGSQGNLIF,TRAV12-2,TRAJ42,HomoSapiens,HLA-A*02,B2M,MHCI,...,IYS----NGD,SYD----VKM,CAVTHYGGSQGNLIF,CASLRSAVWADTQYF,15,15,15,"C, A, V, T, H, Y, G, G, -, -, -, -, S, Q, G, N...",15,"C, A, S, L, R, S, A, V, -, -, -, -, W, A, D, T..."
4,17,18,TRA,CAGGGGGADGLTF,TRAV12-2,TRAJ45,HomoSapiens,HLA-A*02,B2M,MHCI,...,IYS----NGD,SYD----VKM,CAGGGGG--ADGLTF,CASTLTG-LGQPQHF,15,15,15,"C, A, G, G, G, G, G, -, -, -, -, -, -, A, D, G...",15,"C, A, S, T, L, T, G, -, -, -, -, -, L, G, Q, P..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24520,28936,29239,TRA,CAVMRGTALIF,TRAV8-4,TRAJ15,HomoSapiens,HLA-B*07:02,B2M,MHCI,...,YTSA--ATLV,ANQG---SEA,CAVMRG----TALIF,CSVVGTG-GPETQYF,15,15,15,"C, A, V, M, R, G, -, -, -, -, -, -, -, -, T, A...",15,"C, S, V, V, G, T, G, -, -, -, -, -, G, P, E, T..."
24521,28937,29240,TRA,CAVPTYSSASKIIF,TRAV8-4,TRAJ3,HomoSapiens,HLA-B*07:02,B2M,MHCI,...,YTSA--ATLV,FQG----NSA,CAVPTYS-SASKIIF,CASSSG---GYEQYF,15,15,15,"C, A, V, P, T, Y, S, -, -, -, -, -, S, A, S, K...",15,"C, A, S, S, S, G, -, -, -, -, -, -, -, G, Y, E..."
24522,28938,29241,TRA,CALSATSGTYKYIF,TRAV9-2,TRAJ40,HomoSapiens,HLA-B*07:02,B2M,MHCI,...,ATKA---DDK,SMN----VEV,CALSATS-GTYKYIF,CASSPLSGTSATKETQYF,15,18,15,"C, A, L, S, A, T, S, -, -, -, -, -, G, T, Y, K...",18,"C, A, S, S, P, L, S, G, T, -, S, A, T, K, E, T..."
24523,28939,29242,TRA,CAALAGTASKLTF,TRAV13-1,TRAJ44,HomoSapiens,HLA-B*07:02,B2M,MHCI,...,IRSN---VGE,ANQG---SEA,CAALAGT--ASKLTF,CSVVPLA-GPYEQYF,15,15,15,"C, A, A, L, A, G, T, -, -, -, -, -, -, A, S, K...",15,"C, S, V, V, P, L, A, -, -, -, -, -, G, P, Y, E..."


In [10]:
vdjdb_no_small_study['Epitope'].value_counts()

GILGFVFTL     1853
AVFDRKSDAK    1699
RAKFKQLL      1200
IVTDFSVIK      704
RLRAEAQVK      412
ELAGIGILTV     374
SSLENFRAYV     349
NLVPMVATV      348
GLCTLVAML      342
YLQPRTFLL      331
HGIRNASFI      243
TTDPSFLGRY     242
LLWNGPMAV      235
CINGVCWTV      226
ASNENMETM      196
SSYRRPVGI      177
SPRWYFYYL      175
SSPPMFRV       133
LSLRNPILV      127
ATDALMTGF      125
LTDEMIAQY      124
KSKRTPMGF       98
Name: Epitope, dtype: int64

In [11]:
vdjdb_no_small_study[['Epitope','Reference']].value_counts()

Epitope     Reference                                          
AVFDRKSDAK  10xGenomics                                            1692
GILGFVFTL   10xGenomics                                            1222
RAKFKQLL    10xGenomics                                            1200
IVTDFSVIK   10xGenomics                                             697
RLRAEAQVK   10xGenomics                                             412
SSLENFRAYV  PMID:28636592                                           349
ELAGIGILTV  10xGenomics                                             325
HGIRNASFI   PMID:28636592                                           243
TTDPSFLGRY  https://github.com/antigenomics/vdjdb-db/issues/326     242
ASNENMETM   PMID:28636592                                           196
YLQPRTFLL   https://github.com/antigenomics/vdjdb-db/issues/326     196
LLWNGPMAV   PMID:28103239                                           187
GILGFVFTL   PMID:28636592                                           186


In [12]:
vdjdb_no_small_study.to_csv('../data/vdj_cleaned_subset_for_MI_no-small-study.csv')